In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Configurações iniciais
np.random.seed(42)
n_leads = 3000
hoje = datetime(2026, 5, 20) # Data de referência do projeto

# 1. Gerar Tabela de Leads (Estática)
lead_ids = [f'L{str(i).zfill(5)}' for i in range(1, n_leads + 1)]
origens = ['Inbound', 'Outbound', 'Parceiros', 'Eventos']
portes = ['Pequena', 'Média', 'Grande', 'Enterprise']

df_leads = pd.DataFrame({
    'lead_id': lead_ids,
    'data_criacao': [hoje - timedelta(days=np.random.randint(10, 180)) for _ in range(n_leads)],
    'origem': np.random.choice(origens, n_leads, p=[0.5, 0.3, 0.1, 0.1]),
    'porte_empresa': np.random.choice(portes, n_leads, p=[0.4, 0.35, 0.15, 0.1]),
    'status_atual': np.random.choice(['Aberto', 'Ganho', 'Perdido'], n_leads, p=[0.5, 0.2, 0.3])
})

# 2. Gerar Tabela de Eventos (Log Transacional)
tipos_evento = [
    'email_aberto', 
    'email_clicado', 
    'site_visita_pricing', 
    'download_ebook', 
    'reuniao_agendada'
]
pesos_eventos = [0.4, 0.2, 0.15, 0.15, 0.1]

eventos = []
for _, lead in df_leads.iterrows():
    # Leads 'Ganhos' tendem a ter mais interações
    n_eventos = np.random.randint(1, 15) if lead['status_atual'] == 'Ganho' else np.random.randint(0, 8)
    
    for _ in range(n_eventos):
        # Eventos ocorrem entre a criação do lead e a data de hoje
        dias_desde_criacao = (hoje - lead['data_criacao']).days
        if dias_desde_criacao > 0:
            dias_atras = np.random.randint(0, dias_desde_criacao)
            data_evento = hoje - timedelta(days=dias_atras)
            
            eventos.append({
                'lead_id': lead['lead_id'],
                'data_evento': data_evento,
                'tipo_evento': np.random.choice(tipos_evento, p=pesos_eventos)
            })

df_eventos = pd.DataFrame(eventos)
df_eventos = df_eventos.sort_values(by=['data_evento']).reset_index(drop=True)

print(f"Leads gerados: {len(df_leads)}")
print(f"Eventos de engajamento gerados: {len(df_eventos)}\n")
print("Amostra do Log de Eventos:")
print(df_eventos.head())

Leads gerados: 3000
Eventos de engajamento gerados: 12724

Amostra do Log de Eventos:
  lead_id data_evento     tipo_evento
0  L00565  2025-11-23   email_clicado
1  L02699  2025-11-24    email_aberto
2  L00219  2025-11-25  download_ebook
3  L00600  2025-11-26    email_aberto
4  L00474  2025-11-26    email_aberto


In [2]:
import threading
import uvicorn
from fastapi import FastAPI, Query
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import requests

# 1. Inicializar a App
app = FastAPI(title="Mock CRM API B2B")

# --- GERAR DADOS EM MEMÓRIA ---
np.random.seed(42)
n_leads = 3000
hoje = datetime(2026, 5, 20)

lead_ids = [f'L{str(i).zfill(5)}' for i in range(1, n_leads + 1)]
df_leads = pd.DataFrame({
    'lead_id': lead_ids,
    'data_criacao': [hoje - timedelta(days=np.random.randint(10, 180)) for _ in range(n_leads)],
    'origem': np.random.choice(['Inbound', 'Outbound', 'Parceiros', 'Eventos'], n_leads, p=[0.5, 0.3, 0.1, 0.1]),
    'porte_empresa': np.random.choice(['Pequena', 'Média', 'Grande', 'Enterprise'], n_leads, p=[0.4, 0.35, 0.15, 0.1]),
    'status_atual': np.random.choice(['Aberto', 'Ganho', 'Perdido'], n_leads, p=[0.5, 0.2, 0.3])
})
df_leads['data_criacao'] = df_leads['data_criacao'].dt.strftime('%Y-%m-%d')

tipos_evento = ['email_aberto', 'email_clicado', 'site_visita_pricing', 'download_ebook', 'reuniao_agendada']
pesos_eventos = [0.4, 0.2, 0.15, 0.15, 0.1]
eventos = []

for _, lead in df_leads.iterrows():
    n_eventos = np.random.randint(1, 15) if lead['status_atual'] == 'Ganho' else np.random.randint(0, 8)
    data_obj = datetime.strptime(lead['data_criacao'], '%Y-%m-%d')
    dias_desde_criacao = (hoje - data_obj).days
    for _ in range(n_eventos):
        if dias_desde_criacao > 0:
            dias_atras = np.random.randint(0, dias_desde_criacao)
            eventos.append({
                'lead_id': lead['lead_id'],
                'data_evento': (hoje - timedelta(days=dias_atras)).strftime('%Y-%m-%d'),
                'tipo_evento': np.random.choice(tipos_evento, p=pesos_eventos)
            })
df_eventos = pd.DataFrame(eventos).sort_values(by=['data_evento']).reset_index(drop=True)

# --- ENDPOINTS ---
@app.get("/api/v1/leads")
def get_leads(skip: int = 0, limit: int = 100):
    return {"total": len(df_leads), "data": df_leads.iloc[skip : skip + limit].to_dict(orient="records")}

@app.get("/api/v1/eventos")
def get_eventos(skip: int = 0, limit: int = 500):
    return {"total": len(df_eventos), "data": df_eventos.iloc[skip : skip + limit].to_dict(orient="records")}

# --- TRUQUE DO SERVIDOR EM BACKGROUND ---
# Configura o uvicorn para rodar numa thread separada
def arrancar_servidor():
    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    server.run()

# Inicia a thread apenas se o servidor já não estiver ativo
server_thread = threading.Thread(target=arrancar_servidor, daemon=True)
server_thread.start()
print("Servidor FastAPI iniciado com sucesso em segundo plano (Porta 8000)!")

Servidor FastAPI iniciado com sucesso em segundo plano (Porta 8000)!
